# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_raw.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_raw.parquet')

In [4]:
X_train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence


In [5]:
X_test.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


# Machine Learning

In [6]:
models = dict(
    lgbm=load_pickle('../models/layer_1/model_lightgbm.pkl'),
    cat=load_pickle('../models/layer_1/model_catboost.pkl'),
    xgb=load_pickle('../models/layer_1/model_xgboost.pkl'),
    hist=load_pickle('../models/layer_1/model_hist_gradient_boosting.pkl'),
    extra=load_pickle('../models/layer_1/model_extra_tree.pkl'),
    rf=load_pickle('../models/layer_1/model_random_forest.pkl'),
    # lda=load_pickle('../models/layer_1/model_lda.pkl'),
    # linear_svc=load_pickle('../models/layer_1/model_linear_svc.pkl'),
    # lg=load_pickle('../models/layer_1/model_logistic_regression.pkl'),
    # mlp=load_pickle('../models/layer_1/model_mlp.pkl'),
    # qda=load_pickle('../models/layer_1/model_qda.pkl'),
    # ridge=load_pickle('../models/layer_1/model_ridge.pkl'),
    # sgd=load_pickle('../models/layer_1/model_sgdclassifier.pkl'),
    # trunsvd_knn=load_pickle('../models/layer_1/model_trunsvd_knn.pkl'),
)

## Train Dataset

In [7]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(model, X_train, y_train.class_encoded, cv=cv, n_jobs=-1, method='predict_proba')
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                         | 0/6 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 17%|█████████████████████████████▎                                                                                                                                                  | 1/6 [10:02<50:14, 602.88s/it]

Predicting Train Dataset cat


 33%|██████████████████████████████████████████████████████████▋                                                                                                                     | 2/6 [22:16<45:18, 679.69s/it]

...

 67%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                          | 4/6 [25:08<09:13, 276.69s/it]

Predicting Train Dataset extra


 83%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                             | 5/6 [27:46<03:53, 233.90s/it]

Predicting Train Dataset rf


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [49:55<00:00, 499.20s/it]


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in models.items():
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

Predicting Test Dataset lgbm
Predicting Test Dataset cat
Predicting Test Dataset xgb
Predicting Test Dataset hist
Predicting Test Dataset extra
Predicting Test Dataset rf


# Saving

In [14]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_one.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_one.parquet')

In [15]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,extra_0,extra_1,extra_2,rf_0,rf_1,rf_2
0,0.999956,0.000041,2.488056e-06,0.999942,0.000036,2.164097e-05,0.999928,0.000068,3.860194e-06,0.999953,0.000044,3.478719e-06,0.930569,0.007605,0.061826,0.999910,0.000056,0.000034
1,0.988684,0.000261,1.105495e-02,0.982394,0.000034,1.757179e-02,0.992420,0.000523,7.057480e-03,0.983714,0.000841,1.544465e-02,0.819961,0.005144,0.174894,0.970685,0.000350,0.028965
2,0.000002,0.999998,9.047677e-08,0.000022,0.999978,1.357429e-07,0.000002,0.999997,1.084726e-07,0.000001,0.999999,9.847311e-08,0.007812,0.961837,0.030351,0.000011,0.999979,0.000011
3,0.999884,0.000113,3.403338e-06,0.999815,0.000182,2.531501e-06,0.999781,0.000213,6.182258e-06,0.999371,0.000627,2.417958e-06,0.943643,0.006641,0.049717,0.999213,0.000571,0.000216
4,0.999361,0.000626,1.369612e-05,0.998781,0.001195,2.377896e-05,0.999596,0.000392,1.163140e-05,0.992451,0.007527,2.222359e-05,0.899693,0.009009,0.091297,0.998640,0.000989,0.000371


In [16]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,extra_0,extra_1,extra_2,rf_0,rf_1,rf_2
0,0.999184,0.000763,0.000053,0.997587,0.001932,0.000480,0.998360,0.001543,0.000097,0.998873,0.001052,0.000075,0.665138,0.066561,0.268301,0.987747,0.006744,0.005509
1,0.997347,0.002648,0.000006,0.998597,0.001402,0.000001,0.998518,0.001476,0.000006,0.991409,0.008583,0.000008,0.948686,0.016400,0.034914,0.997904,0.001933,0.000163
2,0.998253,0.000180,0.001567,0.999326,0.000013,0.000661,0.998020,0.000144,0.001837,0.992865,0.000330,0.006805,0.604564,0.021978,0.373458,0.977949,0.003788,0.018263
3,0.001209,0.000315,0.998475,0.001704,0.000080,0.998216,0.000952,0.000309,0.998739,0.000192,0.000183,0.999626,0.072561,0.055581,0.871859,0.008787,0.004687,0.986526
4,0.999902,0.000093,0.000006,0.999799,0.000198,0.000002,0.999733,0.000259,0.000008,0.999429,0.000553,0.000019,0.956355,0.012354,0.031291,0.999828,0.000095,0.000077


In [17]:
X_train.shape

(577347, 10)

In [18]:
X_test.shape

(247435, 10)

In [19]:
y_train.head()

,class,class_encoded
id,,
0,GALAXY,0
1,GALAXY,0
2,QSO,1
3,GALAXY,0
4,GALAXY,0
